In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('C:/Users/awadh/Wheat/data'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

C:/Users/awadh/Wheat/data\test\aphid_test\aphid_27.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_28.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_29.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_30.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_31.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_32.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_33.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_34.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_35.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_36.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_37.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_38.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_39.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_40.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_41.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_42.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_43.png
C:/Users/awadh/Wheat/data\test\aphid_test\aphid_44.png
C:/Users/a

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import time
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

ModuleNotFoundError: No module named 'torch'

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

# Param

In [4]:
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-4
NUM_CLASSES = 15
IMG_SIZE = 224

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.2, 0.2)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [6]:
train_dataset = datasets.ImageFolder(
    root='/kaggle/input/wheat-plant-diseases/data/train',
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root='/kaggle/input/wheat-plant-diseases/data/valid',
    transform=val_test_transform
)

test_dataset = datasets.ImageFolder(
    root='/kaggle/input/wheat-plant-diseases/data/test',
    transform=val_test_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


# Load Model

In [7]:
class EfficientNetV2(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(EfficientNetV2, self).__init__()
        self.model = models.efficientnet_v2_l(weights=models.EfficientNet_V2_L_Weights.IMAGENET1K_V1)
            
        # Ganti classification head
        num_ftrs = self.model.classifier[1].in_features
        self.model.classifier = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        return self.model(x)

In [8]:
class ConvNeXt(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(ConvNeXt, self).__init__()
        self.model = models.convnext_large(weights=models.ConvNeXt_Large_Weights.IMAGENET1K_V1)
        
        num_ftrs = self.model.classifier[-1].in_features # Ambil in_features dari Linear layer terakhir
        self.model.classifier = nn.Sequential(
            nn.Flatten(start_dim=1),  # Pindahkan ke atas
            nn.LayerNorm((num_ftrs,), eps=1e-06, elementwise_affine=True),
            nn.Linear(num_ftrs, 256),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        return self.model(x)

In [9]:
class ResNet50V2(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(ResNet50V2, self).__init__()
        self.model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
            
        # Ganti classification head
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        return self.model(x)

# Function Train Eval Plot

In [10]:
def train_model(model, model_name, train_loader, val_loader, criterion, optimizer, num_epochs=EPOCHS):
    print(f"\n--- Training {model_name} ---")
    start_time = time.time()
    
    best_val_accuracy = 0.0
    best_model_path = f"best_{model_name}.pth"
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        print(f"Epoch {epoch+1}/{num_epochs}")
        
        # Training phase
        model.train()
        running_loss = 0.0
        running_corrects = 0
        
        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            
        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = running_corrects.double() / len(train_loader.dataset)
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc.item())

        # Validation phase
        model.eval()
        running_loss = 0.0
        running_corrects = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
        
        epoch_val_loss = running_loss / len(val_loader.dataset)
        epoch_val_acc = running_corrects.double() / len(val_loader.dataset)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc.item())
        
        epoch_time = time.time() - epoch_start_time
        print(f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f} | Time: {epoch_time:.2f}s")
        
        # Save best model
        if epoch_val_acc > best_val_accuracy:
            best_val_accuracy = epoch_val_acc
            torch.save(model.state_dict(), best_model_path)

    total_time = time.time() - start_time
    print(f"Training {model_name} complete in {total_time // 60:.0f}m {total_time % 60:.0f}s")
    print(f"Best Val Acc for {model_name}: {best_val_accuracy:.4f}")
    
    return history

In [11]:
def evaluate_model(model, model_name, test_loader, class_names):
    print(f"\n--- Evaluating {model_name} ---")
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1_score, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0
    )
    
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test Precision (weighted): {precision:.4f}")
    print(f"Test Recall (weighted): {recall:.4f}")
    print(f"Test F1-score (weighted): {f1_score:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.savefig(f"confusion_matrix_{model_name}.png")
    plt.close() 
    
    return {
        "model_name": model_name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score
    }

In [12]:
def plot_training_history(history, model_name):
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title(f'Loss vs. Epochs - {model_name}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title(f'Accuracy vs. Epochs - {model_name}')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig(f"training_history_{model_name}.png")
    plt.close()

# Train Model

In [13]:
architectures_to_train = {
    "ConvNeXt": ConvNeXt,
    "ResNet50V2": ResNet50V2
}

In [14]:
all_results = []
class_names = train_dataset.classes
for model_name, model_class in architectures_to_train.items():
    model = model_class(num_classes=NUM_CLASSES).to(device)
    best_model_path = f"best_{model_name}.pth"

    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    history = train_model(model, model_name, train_loader, val_loader, criterion, optimizer, num_epochs=EPOCHS)
    plot_training_history(history, model_name)

    print(f"Loading best model for evaluation.")

    model.load_state_dict(torch.load(best_model_path, map_location=device))
    # Evaluasi model
    eval_metrics = evaluate_model(model, model_name, test_loader, class_names)
    eval_metrics['best_model_path'] = best_model_path 
    all_results.append(eval_metrics)

results_df = pd.DataFrame(all_results)
results_df.to_csv("models_evaluation.csv", index=False)
print("\n--- All models trained and evaluated. ---")
print(results_df)

Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth
100%|██████████| 755M/755M [00:11<00:00, 71.0MB/s]



--- Training ConvNeXt ---
Epoch 1/20
Train Loss: 0.8315 Acc: 0.7404 | Val Loss: 0.9757 Acc: 0.7800 | Time: 1704.65s
Epoch 2/20
Train Loss: 0.4132 Acc: 0.8619 | Val Loss: 0.8935 Acc: 0.8400 | Time: 1575.19s
Epoch 3/20
Train Loss: 0.3118 Acc: 0.8937 | Val Loss: 0.8331 Acc: 0.8667 | Time: 1576.59s
Epoch 4/20
Train Loss: 0.2579 Acc: 0.9113 | Val Loss: 0.7876 Acc: 0.8967 | Time: 1579.38s
Epoch 5/20
Train Loss: 0.2276 Acc: 0.9178 | Val Loss: 0.6744 Acc: 0.9067 | Time: 1575.52s
Epoch 6/20
Train Loss: 0.1977 Acc: 0.9287 | Val Loss: 0.9020 Acc: 0.8900 | Time: 1577.76s
Epoch 7/20
Train Loss: 0.1853 Acc: 0.9325 | Val Loss: 0.8360 Acc: 0.9167 | Time: 1577.45s
Epoch 8/20
Train Loss: 0.1630 Acc: 0.9367 | Val Loss: 0.9475 Acc: 0.9067 | Time: 1577.15s
Epoch 9/20
Train Loss: 0.1680 Acc: 0.9366 | Val Loss: 0.8633 Acc: 0.9167 | Time: 1592.84s
Epoch 10/20
Train Loss: 0.1470 Acc: 0.9436 | Val Loss: 0.8958 Acc: 0.9067 | Time: 1581.37s
Epoch 11/20
Train Loss: 0.1471 Acc: 0.9413 | Val Loss: 0.9037 Acc: 0.903

<ipython-input-14-d0e5d73379ae>:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=device))



--- Evaluating ConvNeXt ---
Test Accuracy: 0.9053
Test Precision (weighted): 0.9375
Test Recall (weighted): 0.9053
Test F1-score (weighted): 0.8892


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 195MB/s]



--- Training ResNet50V2 ---
Epoch 1/20
Train Loss: 1.1615 Acc: 0.6380 | Val Loss: 1.3313 Acc: 0.7033 | Time: 428.11s
Epoch 2/20
Train Loss: 0.6396 Acc: 0.7927 | Val Loss: 1.0029 Acc: 0.7833 | Time: 422.18s
Epoch 3/20
Train Loss: 0.5204 Acc: 0.8301 | Val Loss: 1.2385 Acc: 0.8100 | Time: 433.90s
Epoch 4/20
Train Loss: 0.4479 Acc: 0.8491 | Val Loss: 1.1001 Acc: 0.8067 | Time: 434.34s
Epoch 5/20
Train Loss: 0.3960 Acc: 0.8659 | Val Loss: 1.2451 Acc: 0.8467 | Time: 434.03s
Epoch 6/20
Train Loss: 0.3514 Acc: 0.8809 | Val Loss: 1.1731 Acc: 0.8400 | Time: 432.36s
Epoch 7/20
Train Loss: 0.3232 Acc: 0.8878 | Val Loss: 1.0051 Acc: 0.8567 | Time: 428.33s
Epoch 8/20
Train Loss: 0.2982 Acc: 0.8959 | Val Loss: 0.8847 Acc: 0.8700 | Time: 432.26s
Epoch 9/20
Train Loss: 0.2732 Acc: 0.9042 | Val Loss: 1.6066 Acc: 0.8767 | Time: 427.53s
Epoch 10/20
Train Loss: 0.2542 Acc: 0.9102 | Val Loss: 1.1869 Acc: 0.8800 | Time: 428.24s
Epoch 11/20
Train Loss: 0.2351 Acc: 0.9138 | Val Loss: 1.1039 Acc: 0.9033 | Time

<ipython-input-14-d0e5d73379ae>:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=device))


Test Accuracy: 0.8973
Test Precision (weighted): 0.9279
Test Recall (weighted): 0.8973
Test F1-score (weighted): 0.8802

--- All models trained and evaluated. ---
   model_name  accuracy  precision    recall  f1_score      best_model_path
0    ConvNeXt  0.905333   0.937493  0.905333  0.889193    best_ConvNeXt.pth
1  ResNet50V2  0.897333   0.927920  0.897333  0.880216  best_ResNet50V2.pth
